# Part C4 (Bonus, +10 pts): Attention with OpenAI Triton

Implement one head of scaled dot-product attention with [Triton](https://triton-lang.org/),
the Python-embedded language for GPU kernels you saw in the Advanced CUDA and Triton lecture.
This is the same operation as C1/C2/C3, written the Triton way: you write `tl.load`/`tl.store`
on pointers and `tl.dot` for the matmuls, and the compiler handles the shared memory, the
barriers, and the tensor-core instructions you placed by hand in the CUDA versions.

This is a **bonus** question. It is graded on a rubric by the TA, not by the local grader.
Run it on a GPU runtime (Colab, Insomnia, or your own GPU box).

You will complete two things:
- `attn_kernel`: a `@triton.jit` kernel that computes attention tile by tile with an online softmax.
- `my_attention`: the Python launcher that sets up the grid and calls the kernel.

The notebook checks your result against a PyTorch reference and benchmarks it against
`torch.nn.functional.scaled_dot_product_attention`, whose selected backend is recorded below.

In [1]:
import torch
import triton
import triton.language as tl

assert torch.cuda.is_available(), "This notebook needs a CUDA GPU (run it on Colab/Insomnia with a GPU runtime)."
device = torch.device("cuda")
torch.manual_seed(0)

# Keep the independent reference in FP32 rather than enabling TF32 matmul.
torch.backends.cuda.matmul.allow_tf32 = False
print('GPU:', torch.cuda.get_device_name())
print('PyTorch:', torch.__version__, '| Triton:', triton.__version__, '| CUDA:', torch.version.cuda)


GPU: NVIDIA GeForce RTX 4060 Laptop GPU
PyTorch: 2.7.1+cu126 | Triton: 3.3.1 | CUDA: 12.6


## Problem setup

One attention head. Query, key, and value tensors `Q`, `K`, `V` each have shape `(N, d)`.

- scores `S = (1 / sqrt(d)) * Q @ K.T`, shape `(N, N)`
- weights `P = softmax(S)` row-wise (safe softmax: subtract the row max before `exp`)
- output `O = P @ V`, shape `(N, d)`

We use `float32` here, which is the usual precision for attention kernels and what
`tl.dot` maps onto the tensor cores. The CUDA parts C1/C2/C3 use `float64` for an exact
checksum cross-check; this bonus is checked against a PyTorch reference instead, so the
precision does not need to match them. We pick `N = 2048`, `d = 64`.

In [2]:
N, d = 2048, 64
scale = 1.0 / (d ** 0.5)

Q = torch.randn(N, d, device=device, dtype=torch.float32)
K = torch.randn(N, d, device=device, dtype=torch.float32)
V = torch.randn(N, d, device=device, dtype=torch.float32)

def reference_attention(Q, K, V, scale):
    S = (Q @ K.transpose(0, 1)) * scale
    P = torch.softmax(S, dim=1)
    return P @ V

O_ref = reference_attention(Q, K, V, scale)
print("reference O shape", tuple(O_ref.shape), "checksum", O_ref.sum().item())

reference O shape (2048, 64) checksum 265.43853759765625


## `attn_kernel`

Write a `@triton.jit` kernel that computes one head of scaled dot-product attention:
`O = softmax(scale * Q @ K.T) @ V`, with a numerically stable softmax. `Q`, `K`, `V`, `O`
are row-major `(N, BD)` fp32 tensors passed as pointers with their strides; `BD` is the head
dimension. The hard constraint is that you must not materialize the full `N x N` score
matrix: process the keys and values in tiles of `BN` and carry the softmax state across
tiles. How you map Triton programs to the work, and what you keep per program, is yours to
decide. `tl.dot` is available for the tile matmuls.


In [3]:
@triton.jit
def attn_kernel(
    Q, K, V, O,
    N, scale,
    sqm, sqd, skn, skd, svn, svd, som, sod,
    BM: tl.constexpr, BN: tl.constexpr, BD: tl.constexpr, D_PAD: tl.constexpr,
):
    # One program owns BM query rows; keep only a BM x BN score tile.
    rows = tl.program_id(0) * BM + tl.arange(0, BM)
    cols = tl.arange(0, D_PAD)
    keys = tl.arange(0, BN)
    q = tl.load(Q + rows[:, None] * sqm + cols[None, :] * sqd,
                (rows[:, None] < N) & (cols[None, :] < BD), other=0)
    m = tl.full((BM,), -float('inf'), tl.float32)
    denom = tl.zeros((BM,), tl.float32)
    acc = tl.zeros((BM, D_PAD), tl.float32)
    for start in range(0, N, BN):
        kn = start + keys
        k = tl.load(K + kn[None, :] * skn + cols[:, None] * skd,
                    (kn[None, :] < N) & (cols[:, None] < BD), other=0)
        # TF32x3 retains substantially more FP32 accuracy than a single TF32 dot.
        scores = tl.dot(q, k, input_precision='tf32x3') * scale
        scores = tl.where(kn[None, :] < N, scores, -float('inf'))
        m_new = tl.maximum(m, tl.max(scores, axis=1))
        alpha = tl.exp(m - m_new)
        p = tl.exp(scores - m_new[:, None])
        v = tl.load(V + kn[:, None] * svn + cols[None, :] * svd,
                    (kn[:, None] < N) & (cols[None, :] < BD), other=0)
        # Rescale BOTH accumulated numerator and denominator to the new max.
        acc = acc * alpha[:, None] + tl.dot(p, v, input_precision='tf32x3')
        denom = denom * alpha + tl.sum(p, axis=1)
        m = m_new
    out = acc / denom[:, None]
    tl.store(O + rows[:, None] * som + cols[None, :] * sod, out,
             (rows[:, None] < N) & (cols[None, :] < BD))


## `my_attention`

Write the launcher: allocate the `(N, d)` output, choose the grid and block sizes, and call
`attn_kernel` with the tensors' strides. Return the output tensor.


In [4]:
def my_attention(Q, K, V, scale, BM=64, BN=64):
    if Q.ndim != 2 or Q.shape != K.shape or Q.shape != V.shape:
        raise ValueError('Q, K, V must have matching (N, d) shapes')
    if any(t.dtype != torch.float32 or not t.is_cuda for t in (Q, K, V)):
        raise ValueError('Q, K, V must be CUDA float32 tensors')
    if Q.device != K.device or Q.device != V.device:
        raise ValueError('Q, K, V must be on the same CUDA device')
    n, head_dim = Q.shape
    if n <= 0 or head_dim <= 0:
        raise ValueError('N and d must be positive')
    if any(b < 16 or b & (b - 1) for b in (BM, BN)):
        raise ValueError('BM and BN must be powers of two >= 16')
    # Larger heads need smaller tiles to fit this GPU's shared memory.
    if head_dim > 128:
        raise ValueError('This launcher supports head dimensions up to 128')
    stages = 2
    if head_dim > 64:
        BM, BN, stages = min(BM, 32), min(BN, 32), 1
    O = torch.empty_like(Q)
    with torch.cuda.device(Q.device):
        attn_kernel[(triton.cdiv(n, BM),)](
            Q, K, V, O, n, scale,
            *Q.stride(), *K.stride(), *V.stride(), *O.stride(),
            BM=BM, BN=BN, BD=head_dim, D_PAD=triton.next_power_of_2(max(head_dim, 16)),
            num_warps=4, num_stages=stages,
        )
    return O

O_triton = my_attention(Q, K, V, scale)
err = (O_triton - O_ref).abs().max().item()
print("Triton checksum", O_triton.sum().item(), "| reference", O_ref.sum().item())
print("max abs error vs reference:", err)
assert err < 1e-2, f"Triton output does not match the reference (max abs err {err:.3g})"


Triton checksum 265.4385070800781 | reference 265.43853759765625
max abs error vs reference: 4.023313522338867e-07


## Benchmark against PyTorch SDPA

SDPA selects a backend according to the input dtype, shape, GPU, and installed
PyTorch version. We measure its default selection and inspect the dispatched
operator separately instead of assuming it uses FlashAttention for FP32.
Both functions use the same resident FP32 inputs. `do_bench` warms each function
and uses CUDA events; compilation and input generation are excluded.


In [5]:
from triton.testing import do_bench

def sdpa():
    # SDPA expects (batch, heads, seq, dim); add singleton batch and head dims.
    q = Q[None, None]; k = K[None, None]; v = V[None, None]
    return torch.nn.functional.scaled_dot_product_attention(q, k, v)

import statistics
benchmark_rounds = []
for trial in range(3):
    functions = [('triton_ms', lambda: my_attention(Q, K, V, scale)), ('sdpa_ms', sdpa)]
    if trial % 2:
        functions.reverse()
    measured = {name: do_bench(fn) for name, fn in functions}
    benchmark_rounds.append(measured)
    print('Benchmark round', trial + 1, measured)
ms_triton = statistics.median(r['triton_ms'] for r in benchmark_rounds)
ms_sdpa = statistics.median(r['sdpa_ms'] for r in benchmark_rounds)
print('Reported latency: median of three do_bench mean latencies')
print(f"Triton attention: {ms_triton:.3f} ms")
print(f"PyTorch SDPA:     {ms_sdpa:.3f} ms")
faster, slower = ("SDPA", "Triton") if ms_sdpa < ms_triton else ("Triton", "SDPA")
print(f"{faster} is {max(ms_triton, ms_sdpa) / min(ms_triton, ms_sdpa):.1f}x faster than {slower} at this size")

Benchmark round 1 {'triton_ms': 0.8417280077934265, 'sdpa_ms': 0.5662048634935598}


Benchmark round 2 {'sdpa_ms': 0.454454558306053, 'triton_ms': 0.6165198589626112}


Benchmark round 3 {'triton_ms': 0.6184265704478248, 'sdpa_ms': 0.41565365081324296}
Reported latency: median of three do_bench mean latencies
Triton attention: 0.618 ms
PyTorch SDPA:     0.454 ms
SDPA is 1.4x faster than Triton at this size


In [6]:
# Additional checks: partial tiles, padded heads, non-contiguous inputs, and stability.
validation = []
gen = torch.Generator(device=device).manual_seed(123)
for n_test, d_test, strided, magnitude in [
    (1, 1, False, 1), (63, 32, False, 1), (65, 64, False, 1),
    (129, 19, False, 1), (257, 64, True, 1), (129, 128, False, 1),
    (129, 64, False, 10),
]:
    inputs = [torch.randn(n_test, d_test * (2 if strided else 1),
                           generator=gen, device=device) for _ in range(3)]
    if strided:
        inputs = [t[:, ::2] for t in inputs]
    qt, kt, vt = inputs
    qt = qt * magnitude if magnitude != 1 else qt
    kt = kt * magnitude if magnitude != 1 else kt
    st = d_test ** -0.5
    actual = my_attention(qt, kt, vt, st)
    expected = reference_attention(qt, kt, vt, st)
    error = (actual - expected).abs().max().item()
    assert torch.isfinite(actual).all() and error < 1e-2, (n_test, d_test, error)
    validation.append(dict(N=n_test, d=d_test, strided=strided,
                           magnitude=magnitude, max_abs_error=error))
    print('PASS', validation[-1])

# Inspect dispatch outside timing. SDPA is also checked against the same reference.
sdpa_error = (sdpa()[0, 0] - O_ref).abs().max().item()
assert sdpa_error < 1e-2
with torch.profiler.profile(activities=[torch.profiler.ProfilerActivity.CPU]) as prof:
    sdpa()
    torch.cuda.synchronize()
backend_ops = [event.key for event in prof.key_averages()
               if 'scaled_dot_product' in event.key]
print('SDPA operators:', backend_ops)
print('SDPA max abs error:', sdpa_error)

import json
results = dict(N=N, d=d, dtype=str(Q.dtype), BM=64, BN=64, num_warps=4, num_stages=2,
               gpu=torch.cuda.get_device_name(), torch=torch.__version__,
               triton=triton.__version__, cuda=torch.version.cuda,
               checksum=O_triton.sum().item(), reference_checksum=O_ref.sum().item(),
               max_abs_error=err, triton_ms=ms_triton, sdpa_ms=ms_sdpa,
               sdpa_max_abs_error=sdpa_error, backend_ops=backend_ops, validation=validation,
               benchmark_rounds=benchmark_rounds)
print('C4_RESULT=' + json.dumps(results))

PASS {'N': 1, 'd': 1, 'strided': False, 'magnitude': 1, 'max_abs_error': 0.0}


PASS {'N': 63, 'd': 32, 'strided': False, 'magnitude': 1, 'max_abs_error': 5.960464477539062e-07}
PASS {'N': 65, 'd': 64, 'strided': False, 'magnitude': 1, 'max_abs_error': 1.1920928955078125e-06}
PASS {'N': 129, 'd': 19, 'strided': False, 'magnitude': 1, 'max_abs_error': 3.5762786865234375e-07}


PASS {'N': 257, 'd': 64, 'strided': True, 'magnitude': 1, 'max_abs_error': 5.662441253662109e-07}
PASS {'N': 129, 'd': 128, 'strided': False, 'magnitude': 1, 'max_abs_error': 8.344650268554688e-07}
PASS {'N': 129, 'd': 64, 'strided': False, 'magnitude': 10, 'max_abs_error': 6.277859210968018e-05}


SDPA operators: ['aten::scaled_dot_product_attention', 'aten::_scaled_dot_product_efficient_attention']
SDPA max abs error: 4.917383193969727e-07
C4_RESULT={"N": 2048, "d": 64, "dtype": "torch.float32", "BM": 64, "BN": 64, "num_warps": 4, "num_stages": 2, "gpu": "NVIDIA GeForce RTX 4060 Laptop GPU", "torch": "2.7.1+cu126", "triton": "3.3.1", "cuda": "12.6", "checksum": 265.4385070800781, "reference_checksum": 265.43853759765625, "max_abs_error": 4.023313522338867e-07, "triton_ms": 0.6184265704478248, "sdpa_ms": 0.454454558306053, "sdpa_max_abs_error": 4.917383193969727e-07, "backend_ops": ["aten::scaled_dot_product_attention", "aten::_scaled_dot_product_efficient_attention"], "validation": [{"N": 1, "d": 1, "strided": false, "magnitude": 1, "max_abs_error": 0.0}, {"N": 63, "d": 32, "strided": false, "magnitude": 1, "max_abs_error": 5.960464477539062e-07}, {"N": 65, "d": 64, "strided": false, "magnitude": 1, "max_abs_error": 1.1920928955078125e-06}, {"N": 129, "d": 19, "strided": false,

## What to submit

Submit this completed notebook (`c4_triton.ipynb`) with `attn_kernel` and `my_attention` filled in,
the correctness check passing, and the benchmark cell run. In your report, note the latency
of your Triton kernel versus PyTorch SDPA, say which was faster at this size, and explain why
(dispatch overhead at small sizes, versus the fused backend's tuned block sizes and
tensor-core use at large sizes).